# sci_run117 — journal-extension experiments

Single, self-contained analysis notebook for the journal version of the paper
(IEEE Transactions on Big Data submission). It adds, on top of the conference
pipeline (`icml_run110_ieeebigdatasubmit.ipynb`): the information-set ladder,
the seven-learner / sample-size triangulation, the fourteen-candidate selection
transfer at two cohort scales (16,000 / 240,000), the event-study cross-check,
and the horizon sweep h in {6, 12, 24}.

How it runs:

* every analysis cell is **load-only**; the single build cell (`[117MAKE]`)
  creates the standardized r240k cohort cache once and announces itself —
  on later runs it is skipped ("already present");
* each section writes its outputs to `out/117*.csv` — the exact files mirrored
  in `out_ref117/` of this repository, so every journal table can be checked
  against machine output without running anything;
* the simulator sections run from this repository alone (`cdtlib/`, `tools/`);
  the real-cohort sections additionally require approved access to the national
  employment-insurance extracts, which cannot be redistributed (see README).

Run top-to-bottom; cells that find their outputs already in `out/` skip
themselves, so interrupted runs resume cleanly.


In [ ]:
%%time
# ═══ [117ENV] 독립 실행 환경 + 공용 함수 + 구판 이관 — conference 는 읽기 전용 ═══
# 이 노트북은 run110 커널 없이 sci2026_a 안에서 단독 실행됩니다. cdtlib 와 데이터는
# confieee2026_a 에서 **읽기만** 하고, 산출물·캐시는 전부 sci2026_a 에만 씁니다.
import os, sys, time, json, gc
import numpy as np, pandas as pd
from scipy import stats as _st
from sklearn.ensemble import HistGradientBoostingRegressor as _HGB
from sklearn.ensemble import RandomForestRegressor as _RF, ExtraTreesRegressor as _ET
from sklearn.linear_model import Ridge as _RD
from sklearn.neighbors import KNeighborsRegressor as _KN
from sklearn.neural_network import MLPRegressor as _ML
from sklearn.preprocessing import StandardScaler as _SS
from sklearn.pipeline import make_pipeline as _MP
from sklearn.model_selection import cross_val_predict as _cvp

RUN_240K          = True        # r240k parquet 이 없으면 어차피 자동 건너뜁니다
RUN_240K_TRANSFER = True        # [117F24] 선택 전이 — 가장 긴 부분(3–7시간)
SKIP_240K_CANDS   = ()          # 예: ('kNN-T 25', 'kNN-T 100')  시간이 모자라면 제외
FORCE_RERUN       = ()          # 예: ('117A', '117C16')  기존 산출물을 무시하고 다시 계산
SEEDS_16K  = (0, 1, 2, 3, 4)    # 논문(run110) 프로토콜 그대로
SEEDS_240K = (0, 1, 2)          # run110 [108] 전례: 표본이 커지면 시드 분산↓ → 시드 축소
SIM_CAND_FILE = '117D_후보표_sim.csv'

_CONF_CANDS = [
    r'D:\dd\class\_coding\p_2026icml\confieee2026_a',          # Windows
    os.path.expanduser('~/p_2026icml/confieee2026_a'),          # Ubuntu(예시)
    os.path.join(os.path.dirname(os.getcwd()), 'confieee2026_a'),  # 형제 폴더
]
PATH_CONF_A = next((p for p in _CONF_CANDS if os.path.isdir(os.path.join(p, 'cdtlib'))), None)
assert PATH_CONF_A, ('[117ENV] confieee2026_a 를 찾지 못했습니다. 위 _CONF_CANDS 에 '
                     '실제 경로를 한 줄 추가하십시오.')
if PATH_CONF_A not in sys.path:
    sys.path.insert(0, PATH_CONF_A)
from cdtlib import config as _cfg
from cdtlib import dataio as dio, outcomes as oc, models as mdl
DIR_SYNT = _cfg.DIR_SYNT                          # (읽기) 합성데이터 — conference 안

PATH_SCI = os.getcwd()
assert 'sci2026_a' in PATH_SCI.replace('\\', '/'), \
    '[117ENV] 이 노트북은 sci2026_a 폴더 안에서 실행하십시오 (현재: %s)' % PATH_SCI
DIR_OUT   = os.path.join(PATH_SCI, 'out');   os.makedirs(DIR_OUT, exist_ok=True)
DIR_CACHE = os.path.join(PATH_SCI, 'cache'); os.makedirs(DIR_CACHE, exist_ok=True)

REPORT, _T0 = {}, time.time()
def step(msg):
    print('◆ %s  (+%.0fs)' % (msg, time.time() - _T0), flush=True)
def save_csv(df, name):
    df.to_csv(os.path.join(DIR_OUT, name), index=False, encoding='utf-8-sig')
def have(fname, key):
    # 산출물이 이미 있으면 읽어 돌려주고(셀 건너뛰기), 없거나 FORCE_RERUN 이면 None
    _p = os.path.join(DIR_OUT, fname)
    if key in FORCE_RERUN or not os.path.exists(_p):
        return None
    print('  [%s] 기존 산출물 재사용: %s  (다시 계산하려면 FORCE_RERUN 에 %r 추가)'
          % (key, fname, key))
    return pd.read_csv(_p, encoding='utf-8-sig')

# ── 구판(run111/113) 산출물 자동 이관 — 유효한 것만.
#    111 의 실데이터 결과(111C · 111D 의 real_Qini · 111E 의 실코호트 행)는
#    2,000명 코호트 버그의 산물이라 이관하지 않고 이 노트북이 새로 계산합니다.
def _migrate():
    _M = [
        # run116 산출물 최우선 이관
        ('116A_사다리.csv',        '117A_사다리.csv',        None),
        ('116B_학습기삼각.csv',    '117B_학습기삼각.csv',    None),
        ('116B_n스케일링.csv',     '117B_n스케일링.csv',     None),
        ('116D_후보표_sim.csv',    SIM_CAND_FILE,            None),
        ('116E_지평_sim.csv',      '117E_지평_sim.csv',      None),
        ('116C_사건연구_r16k.csv',  '117C_사건연구_r16k.csv',  None),
        ('116D_후보표_r16k.csv',    '117D_후보표_r16k.csv',    None),
        ('116D_전이검정_r16k.csv',  '117D_전이검정_r16k.csv',  None),
        ('116E_지평_r16k.csv',      '117E_지평_r16k.csv',      None),
        ('116C_사건연구_r240k.csv', '117C_사건연구_r240k.csv', None),
        ('116E_지평_r240k.csv',     '117E_지평_r240k.csv',     None),
        ('116F_후보표_r240k.csv',   '117F_후보표_r240k.csv',   None),
        ('116F_전이검정_교차규모.csv', '117F_전이검정_교차규모.csv', None),
        # run115 산출물 이관
        ('115A_사다리.csv',        '117A_사다리.csv',        None),
        ('115B_학습기삼각.csv',    '117B_학습기삼각.csv',    None),
        ('115B_n스케일링.csv',     '117B_n스케일링.csv',     None),
        ('115D_후보표_sim.csv',    SIM_CAND_FILE,            None),
        ('115E_지평_sim.csv',      '117E_지평_sim.csv',      None),
        ('115C_사건연구_r16k.csv',  '117C_사건연구_r16k.csv',  None),
        ('115D_후보표_r16k.csv',    '117D_후보표_r16k.csv',    None),
        ('115D_전이검정_r16k.csv',  '117D_전이검정_r16k.csv',  None),
        ('115E_지평_r16k.csv',      '117E_지평_r16k.csv',      None),
        ('115C_사건연구_r240k.csv', '117C_사건연구_r240k.csv', None),
        ('115E_지평_r240k.csv',     '117E_지평_r240k.csv',     None),
        ('115F_후보표_r240k.csv',   '117F_후보표_r240k.csv',   None),
        ('115F_전이검정_교차규모.csv', '117F_전이검정_교차규모.csv', None),
        # run114 산출물 이관 (있으면 그대로 117 이름으로)
        ('114A_사다리.csv',        '117A_사다리.csv',        None),
        ('114B_학습기삼각.csv',    '117B_학습기삼각.csv',    None),
        ('114B_n스케일링.csv',     '117B_n스케일링.csv',     None),
        ('114D_후보표_sim.csv',    SIM_CAND_FILE,            None),
        ('114E_지평_sim.csv',      '117E_지평_sim.csv',      None),
        ('114C_사건연구_r16k.csv',  '117C_사건연구_r16k.csv',  None),
        ('114D_후보표_r16k.csv',    '117D_후보표_r16k.csv',    None),
        ('114D_전이검정_r16k.csv',  '117D_전이검정_r16k.csv',  None),
        ('114E_지평_r16k.csv',      '117E_지평_r16k.csv',      None),
        ('114C_사건연구_r240k.csv', '117C_사건연구_r240k.csv', None),
        ('114E_지평_r240k.csv',     '117E_지평_r240k.csv',     None),
        ('114F_후보표_r240k.csv',   '117F_후보표_r240k.csv',   None),
        ('114F_전이검정_교차규모.csv', '117F_전이검정_교차규모.csv', None),
        # run111/113 산출물 (114 것이 없을 때의 빈자리 채움)
        ('111A_사다리.csv',        '117A_사다리.csv',        None),
        ('111B_학습기삼각.csv',    '117B_학습기삼각.csv',    None),
        ('111B_n스케일링.csv',     '117B_n스케일링.csv',     None),
        ('111D_후보표.csv',        SIM_CAND_FILE,            'drop_real'),
        ('111E_지평.csv',          '117E_지평_sim.csv',      'sim_rows'),
        ('113C_사건연구_r16k.csv',  '117C_사건연구_r16k.csv',  None),
        ('113D_후보표_r16k.csv',    '117D_후보표_r16k.csv',    None),
        ('113E_지평_r16k.csv',      '117E_지평_r16k.csv',      None),
        ('113C_사건연구_r240k.csv', '117C_사건연구_r240k.csv', None),
        ('113E_지평_r240k.csv',     '117E_지평_r240k.csv',     None),
        ('113F_후보표_r240k.csv',   '117F_후보표_r240k.csv',   None),
    ]
    for _o, _n, _tx in _M:
        _po, _pn = os.path.join(DIR_OUT, _o), os.path.join(DIR_OUT, _n)
        if os.path.exists(_pn) or not os.path.exists(_po):
            continue
        _d = pd.read_csv(_po, encoding='utf-8-sig')
        if _tx == 'drop_real':
            _d = _d.drop(columns=[c for c in ('real_Qini',) if c in _d.columns])
        elif _tx == 'sim_rows':
            _d = _d[_d['판'] == '시뮬레이터']
            _d = _d.drop(columns=[c for c in ('Qini',) if c in _d.columns])
        _d.to_csv(_pn, index=False, encoding='utf-8-sig')
        print('  [이관] %s → %s' % (_o, _n))
_migrate()

# ── 코호트 로더 — run111 버그(use_cohort 누락 → 옛 2,000명)의 재발을 구조로 막습니다
_N_EXPECT = {'r16k': 16000, 'r240k': 240000}

def _std_cache_load(path):
    _z = np.load(path, allow_pickle=False)
    return dio.StdData(meta=json.loads(str(_z['meta'])),
                       **{k: _z[k] for k in dio.ARRAY_KEYS})

def _std_cache_path(tag):
    return os.path.join(DIR_CACHE, 'empservicereal_std_%s.npz' % tag)

def std_available(tag):
    # 이 노트북이 로드할 수 있는 표준화본이 sci2026_a/cache 에 있는가 (conference 는 안 봄)
    return os.path.exists(_std_cache_path(tag))
    
def load_cohort(tag):
    # ★ 로드 전용 — 이 노트북은 데이터를 제작하지 않습니다. 표준화본이 없으면 멈춥니다.
    _cfg.use_cohort(tag)
    if not os.path.exists(_std_cache_path(tag)):
        raise FileNotFoundError(
            '[load_cohort] %s 표준화본이 cache 에 없습니다 — [117MAKE] 셀을 먼저 실행하십시오.' % tag)
    print('  [load_cohort] %s 표준화본을 cache 에서 불러옵니다.' % tag, flush=True)
    s = _std_cache_load(_std_cache_path(tag))
    
    assert s.N == _N_EXPECT[tag], ('[load_cohort] %s 선택 후에도 N=%d — 코호트 파일을 '
                                   '확인하십시오.' % (tag, s.N))
    # 결과묶음 — run110 [105N·고침 G]의 교훈: 코호트 꼬리표가 붙고 길이가 맞는 캐시만 신뢰
    _pc = os.path.join(DIR_CACHE, 'empservicereal_outcomes_%s.npz' % tag)
    o = None
    if os.path.exists(_pc):
        _z = np.load(_pc, allow_pickle=False)
        o = {k: _z[k] for k in _z.files}
        if len(np.asarray(o['TREATED']).reshape(-1)) != s.N:
            print('  [load_cohort] 캐시 길이 불일치 — 버리고 다시 만듭니다:', _pc); o = None
    if o is None:
        o = oc.build(s, 'empservicereal')           # t* 규약 = 논문 §III (고정 시드)
        np.savez_compressed(_pc, **o)
    sp = np.asarray(s.SPLIT); sp = (sp[:, 0] if sp.ndim > 1 else sp).reshape(-1)
    co = dict(tag=tag, s=s, o=o, N=int(s.N), T=int(s.T),
              tr=np.asarray(o['TREATED'], bool).reshape(-1),
              ts=np.asarray(o['TSTAR'], int).reshape(-1), sp=sp)
    print('  [load_cohort] %s: N=%s · 처치 %s · train/valid/test = %d/%d/%d'
          % (tag, format(co['N'], ','), format(int(co['tr'].sum()), ','),
             int((sp == 0).sum()), int((sp == 1).sum()), int((sp == 2).sum())))
    return co

# ── 사건연구 (논문 §III 의 t*·위약 규약 그대로 · 부트스트랩은 복제 chunk 로 메모리 상한)
def event_study(co, B=1000, chunk=100, seed=20260821):
    N, T = co['N'], co['T']
    _em = np.asarray(co['s'].M, bool).reshape(N, T)
    _wg = np.asarray(co['s'].Y[..., 0]).reshape(N, T)
    _y3m = np.where(_em, np.expm1(_wg), 0.0).astype(np.float32)
    KS = np.arange(-24, 25)
    P = np.full((N, len(KS)), np.nan, np.float32)
    for _j, _k in enumerate(KS):
        _m = co['ts'] + _k
        _in = (_m >= 0) & (_m < T)
        P[_in, _j] = _y3m[np.where(_in)[0], _m[_in]]
    _pre = np.nanmean(P[:, (KS >= -24) & (KS <= -13)], axis=1)
    _dm = P - _pre[:, None]
    _it, _ic = np.where(co['tr'])[0], np.where(~co['tr'])[0]

    def _boot_means(V, rng):                       # V: (n, K) → (B, K) 재표집 평균
        out = np.empty((B, V.shape[1]), np.float32)
        done = 0
        while done < B:
            c = min(chunk, B - done)
            R = rng.integers(0, V.shape[0], (c, V.shape[0]))
            for j in range(V.shape[1]):
                out[done:done + c, j] = np.nanmean(V[:, j][R], axis=1)
            done += c
        return out

    def _rows(Q, tag):
        rng = np.random.default_rng(seed)          # 판마다 같은 재표집 규약
        bt, bc = _boot_means(Q[_it], rng), _boot_means(Q[_ic], rng)
        rows = []
        for _j, _k in enumerate(KS):
            _vt, _vc = Q[_it, _j], Q[_ic, _j]
            _bs = bt[:, _j] - bc[:, _j]
            rows.append(dict(판=tag, k=int(_k),
                             처치=float(np.nanmean(_vt)), 대조=float(np.nanmean(_vc)),
                             차이=float(np.nanmean(_vt) - np.nanmean(_vc)),
                             lo=float(np.nanpercentile(_bs, 2.5)),
                             hi=float(np.nanpercentile(_bs, 97.5)),
                             n처치=int(np.isfinite(_vt).sum()), n대조=int(np.isfinite(_vc).sum())))
        return rows
    return pd.DataFrame(_rows(P, '수준') + _rows(_dm, '사전평균제거'))

def es_report(df):
    _d0 = df[df['판'] == '사전평균제거'].set_index('k')
    return 'k=-18 %.2f → k=-1 %.2f → k=0 %.2f → k=+14 %.2f (사전평균제거, 만원)' % (
        _d0.loc[-18, '차이'], _d0.loc[-1, '차이'], _d0.loc[0, '차이'], _d0.loc[14, '차이'])

# ── T-러너 (예측은 필요한 행만 — 값은 전행 예측과 동일, 240k 에서 큰 절약)
def tlearn_tau(fac, F, y, trn, trt, seed, pred):
    _m1 = fac(s=seed).fit(F[trn & trt], y[trn & trt])
    _m0 = fac(s=seed).fit(F[trn & ~trt], y[trn & ~trt])
    return _m1.predict(F[pred]) - _m0.predict(F[pred])

def qini_seeds(co, F, y, trn, te, fac, seeds):
    return [float(mdl.qini(tlearn_tau(fac, F, y, trn, co['tr'], _sd, te),
                           co['tr'][te], y[te])) for _sd in seeds]

# ── 후보 14종 — sim 지표([117D1])와 real Qini([117D16]/[117F24])가 같은 목록을 씁니다
def mk_cands():
    _c = []
    for _d in (3, 6, 9):
        for _lr in (0.05, 0.1):
            _c.append(('HGB-T d%d lr%.2f' % (_d, _lr),
                       lambda d=_d, lr=_lr, s=0: _HGB(max_depth=d, learning_rate=lr,
                                                      max_iter=300, random_state=s)))
    _c.append(('RF-T 400',  lambda s=0: _RF(n_estimators=400, min_samples_leaf=5,
                                            random_state=s, n_jobs=-1)))
    _c.append(('RF-T d12',  lambda s=0: _RF(n_estimators=400, max_depth=12,
                                            min_samples_leaf=5, random_state=s, n_jobs=-1)))
    _c.append(('ET-T 400',  lambda s=0: _ET(n_estimators=400, min_samples_leaf=5,
                                            random_state=s, n_jobs=-1)))
    _c.append(('Ridge-T',   lambda s=0: _MP(_SS(), _RD(alpha=1.0))))
    _c.append(('kNN-T 25',  lambda s=0: _MP(_SS(), _KN(n_neighbors=25))))
    _c.append(('kNN-T 100', lambda s=0: _MP(_SS(), _KN(n_neighbors=100))))
    _c.append(('MLP-T 64',  lambda s=0: _MP(_SS(), _ML(hidden_layer_sizes=(64,),
                                            max_iter=400, early_stopping=True, random_state=s))))
    _c.append(('MLP-T 128-64', lambda s=0: _MP(_SS(), _ML(hidden_layer_sizes=(128, 64),
                                            max_iter=400, early_stopping=True, random_state=s))))
    return _c

def perm_p(x, y, n=20000, seed=0):
    _r0 = _st.spearmanr(x, y).statistic
    _rg = np.random.default_rng(seed)
    _cnt = sum(abs(_st.spearmanr(x, _rg.permutation(y)).statistic) >= abs(_r0) - 1e-12
               for _ in range(n))
    return float(_r0), (1 + _cnt) / (1 + n)

def transfer_test(df, qcol):
    # 후보표(sim_rPEHE·sim_rho·qcol) 에서 Spearman + 순열 p + LOO 범위 표를 만든다
    _pehe, _rho = df['sim_rPEHE'].to_numpy(), df['sim_rho'].to_numpy()
    _q = df[qcol].to_numpy()
    _rp, _pp = perm_p(_pehe, _q); _rr, _pr = perm_p(_rho, _q)
    _looP = [_st.spearmanr(np.delete(_pehe, i), np.delete(_q, i)).statistic for i in range(len(df))]
    _looR = [_st.spearmanr(np.delete(_rho, i), np.delete(_q, i)).statistic for i in range(len(df))]
    return pd.DataFrame([
        dict(선택통계='sim √PEHE', 후보수=len(df), Spearman=round(_rp, 3), 순열p=round(_pp, 4),
             LOO범위='[%.2f, %.2f]' % (min(_looP), max(_looP))),
        dict(선택통계='sim ρ',    후보수=len(df), Spearman=round(_rr, 3), 순열p=round(_pr, 4),
             LOO범위='[%.2f, %.2f]' % (min(_looR), max(_looR))),
    ])

# ── 선택 전이의 real Qini 블록 — 후보마다 저장(중단 내성) + 재실행 시 이어돌기
def transfer_block(co, F, seeds, fname_cand, skip=()):
    _sim = pd.read_csv(os.path.join(DIR_OUT, SIM_CAND_FILE), encoding='utf-8-sig')
    _CANDS = mk_cands()
    assert [n for n, _ in _CANDS] == list(_sim['후보']), \
        '[transfer_block] 후보 목록이 %s 와 다릅니다 — mk_cands 를 바꾸지 마십시오.' % SIM_CAND_FILE
    _qcol = 'real_Qini_%s' % co['tag']
    _done = {}
    _pf = os.path.join(DIR_OUT, fname_cand)
    if os.path.exists(_pf):                        # 이어돌기
        _old = pd.read_csv(_pf, encoding='utf-8-sig')
        if _qcol in _old.columns:
            _done = {r['후보']: r for _, r in _old.iterrows() if np.isfinite(r[_qcol])}
        if _done: print('  [이어돌기] 끝난 후보 %d개는 건너뜁니다: %s'
                        % (len(_done), ', '.join(_done)), flush=True)
    _kH = list(np.asarray(co['o']['HORIZONS'])).index(12)
    _y = np.asarray(co['o']['Y3'])[:, _kH].astype(float)
    _ok = np.isfinite(_y)
    _trn, _te = (co['sp'] == 0) & _ok, (co['sp'] == 2) & _ok
    print('  [%s] 학습 %s · 시험 %s (시험 처치 %s) · 시드 %s'
          % (co['tag'], format(int(_trn.sum()), ','), format(int(_te.sum()), ','),
             format(int((_te & co['tr']).sum()), ','), list(seeds)), flush=True)
    _rows = []
    for _nm, _fac in _CANDS:
        if _nm in _done:
            _r = _done[_nm]
            _rows.append(dict(후보=_nm, **{_qcol: _r[_qcol], 'Qini_sd': _r['Qini_sd'],
                                           '분': _r.get('분', np.nan)})); continue
        if _nm in skip:
            print('  %-16s 건너뜀 (SKIP 목록)' % _nm, flush=True)
            _rows.append(dict(후보=_nm, **{_qcol: np.nan, 'Qini_sd': np.nan, '분': np.nan})); continue
        _t0 = time.time()
        _qs = qini_seeds(co, F, _y, _trn, _te, _fac, seeds)
        _min = (time.time() - _t0) / 60
        _rows.append(dict(후보=_nm, **{_qcol: round(float(np.nanmean(_qs)), 2),
                                       'Qini_sd': round(float(np.nanstd(_qs)), 2),
                                       '분': round(_min, 1)}))
        print('  %-16s Qini=%.1f ± %.1f  (%.1f분)' % (_nm, np.nanmean(_qs), np.nanstd(_qs), _min),
              flush=True)
        save_csv(pd.DataFrame(_rows), fname_cand)  # 후보마다 저장 — 끊겨도 남습니다
    D = _sim[['후보', 'sim_rPEHE', 'sim_rho']].merge(pd.DataFrame(_rows), on='후보')
    save_csv(D, fname_cand)
    return D, _qcol

# ── 지평 블록 (실코호트)
def horizons_block(co, F, reps, seeds):
    _HORS = list(np.asarray(co['o']['HORIZONS']))
    _rows = []
    for _h in (6, 12, 24):
        _kh = _HORS.index(_h)
        _yh = np.asarray(co['o']['Y3'])[:, _kh].astype(float); _okh = np.isfinite(_yh)
        _tnh, _teh = (co['sp'] == 0) & _okh, (co['sp'] == 2) & _okh
        for _nm, _fac in reps:
            _qs = qini_seeds(co, F, _yh, _tnh, _teh, _fac, seeds)
            _rows.append(dict(지평=_h, 후보=_nm,
                              **{'Qini_%s' % co['tag']: round(float(np.nanmean(_qs)), 2)},
                              n시험=int(_teh.sum())))
            print('  h=%-2d %-10s Qini=%.1f (n=%s)' % (_h, _nm, np.nanmean(_qs),
                  format(int(_teh.sum()), ',')), flush=True)
    return pd.DataFrame(_rows)

print('[117ENV] conference(읽기):', PATH_CONF_A)
print('[117ENV] 산출물(쓰기)    :', DIR_OUT)
print('[117ENV] 캐시(쓰기)      :', DIR_CACHE)
print('[117ENV] cdtlib 는 참조만 하며, 어떤 셀도 conference 폴더에 쓰지 않습니다.')
step('[117ENV] 환경 준비')

In [ ]:
%%time
# ═══ [117P] 시뮬레이터 공통 준비 — 본판 v2 로드, 관측특징 F0S · 동인 · 참효과 ═══
_pq = os.path.join(DIR_SYNT, 'empservicesynt_v2.parquet')
_tq = os.path.join(DIR_SYNT, 'empservicesynt_v2_truth.parquet')
_eq = os.path.join(DIR_SYNT, 'empservicesynt_v2_person.parquet')
for _f in (_pq, _tq, _eq):
    assert os.path.exists(_f), '[117P] 본판 파일이 없습니다: %s' % _f

_dfS = pd.read_parquet(_pq)
_trjS = pd.read_parquet(_tq)
_perS = pd.read_parquet(_eq).reset_index(drop=True)
NS = len(_perS)
TSIM = int(_dfS['메모리'].max()) if '메모리' in _dfS.columns else int(len(_dfS) // NS)
assert len(_dfS) == NS * TSIM, '[117P] 궤적 길이 불일치: %d != %d×%d' % (len(_dfS), NS, TSIM)

TRT_S = (_perS['훈련직종'] != 'c00').to_numpy()
TST_S = _perS['tstar'].to_numpy().astype(int)

# 참효과 τ_h (지평별) — [104S] 와 동일한 추출: MU_Y3 피벗에서 배정 직종 − c00
def _tau_at(h):
    _t = _trjS[_trjS['horizon'] == h]
    _p = _t.pivot(index='persona', columns='action', values='MU_Y3').reindex(_perS['persona'])
    _cols = list(_p.columns); _M = _p.to_numpy(float)
    _idx = np.array([_cols.index(a) if a in _cols else 0 for a in _perS['훈련직종']])
    return _M[np.arange(NS), _idx] - _p['c00'].to_numpy(float)
TAUS = {h: _tau_at(h) for h in (6, 12, 24)}

# 관측 결과 y3(t*+h) 와 관측특징 F0S — [104S] 간소 파이프라인과 동일 구성
_wgS = _dfS['보상_로그임금'].to_numpy().reshape(NS, TSIM); _emS = (_wgS > 0)
def _y3_at(h):
    _m = np.clip(TST_S + h, 0, TSIM - 1)
    return np.where(_emS[np.arange(NS), _m], np.exp(_wgS[np.arange(NS), _m]), 0.0)
Y3S = {h: _y3_at(h) for h in (6, 12, 24)}

_numS = _dfS.select_dtypes(include=[np.number]).to_numpy().reshape(NS, TSIM, -1)
_t0S = np.clip(TST_S - 1, 0, TSIM - 1)
_peS = np.array([_emS[i, :max(TST_S[i], 1)].mean() for i in range(NS)])
_pwS = np.array([_wgS[i, :max(TST_S[i], 1)].mean() for i in range(NS)])
F0S = np.column_stack([_numS[np.arange(NS), _t0S, :], _peS, _pwS, TST_S]).astype(np.float32)

# 동인(생성기 인적속성) — 사다리의 단계 재료
_sexS = (_perS['성별'].to_numpy() == '여').astype(np.float32)
_ageS = _perS['나이'].to_numpy().astype(np.float32)
_eduS = _perS['학력'].to_numpy().astype(np.float32)
_US = _perS['U'].to_numpy().astype(np.float32)
_VS = _perS['V'].to_numpy().astype(np.float32)
_occS = pd.get_dummies(_perS['훈련직종']).to_numpy().astype(np.float32)

OKS = TRT_S & np.isfinite(TAUS[12])
print('[117P] N=%d T=%d · 훈련군 %d · 채점가능 %d · F0 차원 %d · occ 더미 %d'
      % (NS, TSIM, int(TRT_S.sum()), int(OKS.sum()), F0S.shape[1], _occS.shape[1]))

def audit_sim(F, h=12, seeds=(0, 1, 2), depth=6):
    # 간소 감사([104S] 규약): 오라클 CV-R²(훈련군) + T-러너 √PEHE vs 오라클 상수
    _tau = TAUS[h]; _y = Y3S[h]
    _ok = TRT_S & np.isfinite(_tau)
    _g = _HGB(max_depth=depth, max_iter=300, random_state=0)
    _pr = _cvp(_g, F[_ok], _tau[_ok], cv=5)
    _r2 = 1 - float(np.sum((_tau[_ok] - _pr) ** 2)) / float(np.sum((_tau[_ok] - _tau[_ok].mean()) ** 2))
    _pes, _sds = [], []
    for _sd in seeds:
        _rg = np.random.default_rng(_sd)
        _odr = _rg.permutation(NS); _tn = np.zeros(NS, bool); _tn[_odr[:int(NS * 0.7)]] = True
        _m1 = _HGB(random_state=0).fit(F[_tn & TRT_S], _y[_tn & TRT_S])
        _m0 = _HGB(random_state=0).fit(F[_tn & ~TRT_S], _y[_tn & ~TRT_S])
        _th = _m1.predict(F) - _m0.predict(F)
        _te = (~_tn) & _ok
        _pes.append(float(np.sqrt(np.nanmean((_th[_te] - _tau[_te]) ** 2))))
        _sds.append(float(np.nanstd(_tau[_te])))
    return dict(R2_oracle=round(_r2, 4),
                pehe=round(float(np.mean(_pes)), 3), pehe_sd=round(float(np.std(_pes)), 3),
                oracle_const=round(float(np.mean(_sds)), 3),
                beats=bool(np.mean(_pes) < np.mean(_sds)))
step('[117P] 시뮬레이터 공통 준비')

In [ ]:
%%time
# ═══ [117A] 정보집합 사다리 — 「진단이 다이얼을 추적하는가」 (R1·R5 순환성의 정면 답) ═══
# 관측 이력(L0)에서 참 동인(L4)까지 정보를 단계적으로 공개하며 오라클 R² 와
# oracle-constant 판정을 추적합니다. 기대: R² 단조 증가, 어느 단부터 beats=True.
A117 = have('117A_사다리.csv', '117A')
if A117 is None:
    _L = [
        ('L0 관측 이력(기관 보유)',           F0S),
        ('L1 +인적속성(성별·나이·학력)',       np.column_stack([F0S, _sexS, _ageS, _eduS])),
        ('L2 +훈련직종(계획 정보)',           np.column_stack([F0S, _sexS, _ageS, _eduS, _occS])),
        ('L3 +U(흔적 있는 잠재)',             np.column_stack([F0S, _sexS, _ageS, _eduS, _occS, _US])),
        ('L4 +V(흔적 없는 잠재 = 동인 전체)',  np.column_stack([F0S, _sexS, _ageS, _eduS, _occS, _US, _VS])),
    ]
    _rowsA = []
    for _nm, _F in _L:
        _r = audit_sim(_F); _r = dict(정보집합=_nm, 차원=_F.shape[1], **_r); _rowsA.append(_r)
        print('  %-34s R²=%.4f  √PEHE=%.2f  oracle=%.2f  beats=%s'
              % (_nm, _r['R2_oracle'], _r['pehe'], _r['oracle_const'], _r['beats']), flush=True)
    A117 = pd.DataFrame(_rowsA)
    save_csv(A117, '117A_사다리.csv')
    print()
    print('읽는 법: L0 의 낮은 R² 가 「측정 한계」라면 L1~L4 에서도 낮아야 합니다.')
    print('단조 증가 + 특정 단에서 beats 전환 = 진단이 정보 공개량을 추적한다는 증거이며,')
    print('1% 는 설계의 산물이 아니라 측정값입니다. 전환 단은 「어떤 데이터를 더 모으면')
    print('개인화가 수지가 맞는가」라는 기관용 처방이 됩니다.')
print(A117.to_string(index=False))
_bA = A117['beats'].astype(str).str.lower() == 'true'
REPORT['117A 사다리'] = ' · '.join('%s R²=%.3f %s' % (str(r['정보집합'])[:2], r['R2_oracle'],
                                   'O' if _b else 'X') for (_, r), _b in zip(A117.iterrows(), _bA))
step('[117A] 정보집합 사다리')

In [ ]:
%%time
# ═══ [117B] 천장 추정의 삼각검증 — 학습기 6종 + 표본 스케일링 (R4-1 의 답) ═══
# 「CV-R² 가 낮은 것은 학습기 한계·유한표본 때문일 수 있다」에 대해:
# (a) 귀납편향이 다른 학습기들의 CV-R² 수렴, (b) n 스케일링 곡선의 평탄성.
B117 = have('117B_학습기삼각.csv', '117B')
B117n = have('117B_n스케일링.csv', '117B')
if B117 is None or B117n is None:
    _Fo, _to = F0S[OKS], TAUS[12][OKS]
    def _cvr2(est, X, y, cv=5):
        _pr = _cvp(est, X, y, cv=cv)
        return 1 - float(np.sum((y - _pr) ** 2)) / float(np.sum((y - y.mean()) ** 2))
    _learners = [
        ('HGB(d6)',       _HGB(max_depth=6, max_iter=300, random_state=0)),
        ('HGB(d3)',       _HGB(max_depth=3, max_iter=300, random_state=0)),
        ('RandomForest',  _RF(n_estimators=400, min_samples_leaf=5, random_state=0, n_jobs=-1)),
        ('ExtraTrees',    _ET(n_estimators=400, min_samples_leaf=5, random_state=0, n_jobs=-1)),
        ('Ridge(표준화)',  _MP(_SS(), _RD(alpha=1.0))),
        ('kNN(k=50)',     _MP(_SS(), _KN(n_neighbors=50))),
        ('MLP(64,32)',    _MP(_SS(), _ML(hidden_layer_sizes=(64, 32), max_iter=500,
                                         early_stopping=True, random_state=0))),
    ]
    _rowsB = []
    for _nm, _es in _learners:
        _r2 = _cvr2(_es, _Fo, _to)
        _rowsB.append(dict(학습기=_nm, CV_R2=round(_r2, 4)))
        print('  %-14s CV-R² = %.4f' % (_nm, _r2), flush=True)
    _rowsBn = []
    for _n in sorted(set([1000, 2000, 3000, 5000, int(_Fo.shape[0])])):
        _r2s = []
        for _sd in range(5):
            _ix = np.random.default_rng(_sd).choice(_Fo.shape[0], size=min(_n, _Fo.shape[0]),
                                                    replace=False)
            _r2s.append(_cvr2(_HGB(max_depth=6, max_iter=300, random_state=0), _Fo[_ix], _to[_ix]))
        _rowsBn.append(dict(n=_n, CV_R2_mean=round(float(np.mean(_r2s)), 4),
                            CV_R2_sd=round(float(np.std(_r2s)), 4)))
        print('  n=%5d  CV-R² = %.4f ± %.4f' % (_n, np.mean(_r2s), np.std(_r2s)), flush=True)
    B117 = pd.DataFrame(_rowsB); B117n = pd.DataFrame(_rowsBn)
    save_csv(B117, '117B_학습기삼각.csv'); save_csv(B117n, '117B_n스케일링.csv')
    print()
    print('읽는 법: 학습기들이 모두 같은 자릿수의 낮은 R² 에 수렴하고 n 곡선이 평탄하면,')
    print('「추정된 천장」은 학습기 선택·표본크기의 인공물이 아닙니다. 대조군: 같은 학습기들이')
    print('[117A] 의 L4(동인 공개)에서는 큰 R² 를 냅니다 — 학습기 무능 가설의 기각.')
print(B117.to_string(index=False)); print(); print(B117n.to_string(index=False))
REPORT['117B 삼각검증'] = 'R² 범위 [%.3f, %.3f] · n=%d에서 %.3f' % (
    B117['CV_R2'].min(), B117['CV_R2'].max(),
    int(B117n['n'].iloc[-1]), B117n['CV_R2_mean'].iloc[-1])
step('[117B] 삼각검증')

In [ ]:
%%time
# ═══ [117D1] 선택 전이 (a)단계 — 후보 14개의 sim 지표 (√PEHE · ρ, 3시드 70/30) ═══
# (b)단계([117D16]/[117F24])의 real Qini 와 짝을 이뤄 「시뮬레이터 지표로 고르면
# 실데이터 가치와 어긋나는가」를 검정합니다. 후보마다 저장 — 끊겨도 이어서 돕니다.
_CANDS = mk_cands()
_pfD1 = os.path.join(DIR_OUT, SIM_CAND_FILE)
_doneD1 = {}
if '117D1' not in FORCE_RERUN and os.path.exists(_pfD1):
    _oldD1 = pd.read_csv(_pfD1, encoding='utf-8-sig')
    _doneD1 = {r['후보']: r for _, r in _oldD1.iterrows() if np.isfinite(r['sim_rPEHE'])}
    if _doneD1:
        print('  [이어돌기] sim 지표가 있는 후보 %d개는 건너뜁니다.' % len(_doneD1), flush=True)
_tau12, _y12 = TAUS[12], Y3S[12]
_rowsD1 = []
for _nm, _fac in _CANDS:
    if _nm in _doneD1:
        _r = _doneD1[_nm]
        _rowsD1.append(dict(후보=_nm, sim_rPEHE=_r['sim_rPEHE'], sim_rho=_r['sim_rho'])); continue
    _pes, _rhs = [], []
    for _sd in (0, 1, 2):
        _rg = np.random.default_rng(_sd)
        _odr = _rg.permutation(NS); _tn = np.zeros(NS, bool); _tn[_odr[:int(NS * 0.7)]] = True
        _te = (~_tn) & OKS
        _th = tlearn_tau(_fac, F0S, _y12, _tn, TRT_S, _sd, _te)
        _pes.append(float(np.sqrt(np.nanmean((_th - _tau12[_te]) ** 2))))
        _rhs.append(float(np.corrcoef(_th, _tau12[_te])[0, 1]))
    _rowsD1.append(dict(후보=_nm, sim_rPEHE=round(float(np.mean(_pes)), 3),
                        sim_rho=round(float(np.mean(_rhs)), 4)))
    print('  sim %-16s √PEHE=%.2f ρ=%.3f' % (_nm, np.mean(_pes), np.mean(_rhs)), flush=True)
    save_csv(pd.DataFrame(_rowsD1), SIM_CAND_FILE)
D117s = pd.DataFrame(_rowsD1)
save_csv(D117s, SIM_CAND_FILE)
print(); print(D117s.to_string(index=False))
REPORT['117D1 sim지표'] = '후보 %d개 (전부 T-러너: 군별 적합 → τ̂ = m1 − m0)' % len(D117s)
step('[117D1] 후보 sim 지표')

In [ ]:
%%time
# ═══ [117E1] 지평 전개 — 시뮬레이터 행 (h ∈ {6, 12, 24}) ═══
E117s = have('117E_지평_sim.csv', '117E1')
if E117s is None:
    _rowsE1 = []
    for _h in (6, 12, 24):
        _r = audit_sim(F0S, h=_h)
        _rowsE1.append(dict(지평=_h, 판='시뮬레이터', **_r))
        print('  sim h=%-2d  R²=%.4f √PEHE=%.2f oracle=%.2f beats=%s'
              % (_h, _r['R2_oracle'], _r['pehe'], _r['oracle_const'], _r['beats']), flush=True)
    E117s = pd.DataFrame(_rowsE1)
    save_csv(E117s, '117E_지평_sim.csv')
print(E117s.to_string(index=False))
_bE = E117s['beats'].astype(str).str.lower() == 'true'
REPORT['117E1 지평(sim)'] = ' · '.join('h=%d %s' % (r['지평'], 'O' if _b else 'X')
                                       for (_, r), _b in zip(E117s.iterrows(), _bE))
step('[117E1] 지평 sim')

In [ ]:
%%time
# ═══ [117MAKE] r240k 표준화본 제작 — 없을 때 한 번만 (이 노트북의 유일한 '제작' 셀) ═══
# 분석 셀들은 전부 로드 전용입니다. 제작은 이 셀 하나뿐이고, 하는 일은 r240k parquet 을
# 분석 규격(npz)으로 변환해 sci2026_a/cache 에 저장하는 것입니다 (conference 에는 안 씀).
# datastd 나 cache 에 표준화본이 이미 있으면 아무 일도 하지 않습니다.
# 처음 한 번은 보통 30–60분, 메모리 부족 시 더 — 다른 큰 프로그램은 닫고 돌리십시오.
_TAGS_MAKE = [t for t in ('r16k', 'r240k')
              if (t != 'r240k' or RUN_240K)
              and os.path.exists(_cfg.RAW.get('empservicereal_%s' % t, ''))]
for _tg in _TAGS_MAKE:
    if os.path.exists(_std_cache_path(_tg)):
        print('[117MAKE] %s 표준화본이 cache 에 있습니다 — 만들지 않습니다.' % _tg)
        REPORT['117MAKE ' + _tg] = '기존 cache 사용'
        continue
    print('[117MAKE] ★ %s 표준화본 제작 시작 (%s) ★'
          % (_tg, 'r16k 는 1~3분' if _tg == 'r16k' else '보통 30~60분 · 메모리 부족 시 더'), flush=True)
    gc.collect()
    _t0M = time.time()
    _cfg.use_cohort(_tg)
    _sM = dio.build('empservicereal', realization=0)
    _eM = dio.verify_std(_sM)
    if _eM: raise RuntimeError('[117MAKE] 스키마 위반: %s' % _eM)
    assert _sM.N == _N_EXPECT[_tg], '[117MAKE] N=%d != %d — parquet 을 확인하십시오.' % (_sM.N, _N_EXPECT[_tg])
    np.savez_compressed(_std_cache_path(_tg),
                        meta=json.dumps(dict(_sM.meta), ensure_ascii=False),
                        **{k: getattr(_sM, k) for k in dio.ARRAY_KEYS})
    del _sM; gc.collect()
    print('[117MAKE] 저장: %s (%.0f MB · %.0f분)'
          % (_std_cache_path(_tg), os.path.getsize(_std_cache_path(_tg)) / 1e6,
             (time.time() - _t0M) / 60))
    REPORT['117MAKE ' + _tg] = '제작·저장 (%.0f분)' % ((time.time() - _t0M) / 60)
if not _TAGS_MAKE:
    print('[117MAKE] 만들 것 없음 — parquet 자체가 없거나 RUN_240K=False')
step('[117MAKE] 표준화본 점검')

In [ ]:
%%time
# ═══ [117L16] 실코호트 1부: r16k 로드 + 평탄 특징 — 논문 Results 1–5 의 감사 코호트 ═══
CO16 = load_cohort('r16k')
F16 = mdl.flatten_features(CO16['s'], CO16['o'])
step('[117L16] r16k 로드 (특징 %d열)' % F16.shape[1])

In [ ]:
%%time
# ═══ [117C16] 사건연구 — r16k (R3-1·R4-2 의 답: 사전 추세의 직접 노출) ═══
C117a = have('117C_사건연구_r16k.csv', '117C16')
if C117a is None:
    C117a = event_study(CO16)
    save_csv(C117a, '117C_사건연구_r16k.csv')
    print(C117a[C117a['판'] == '사전평균제거'].round(2).to_string(index=False))
    print()
    print('읽는 법: k<0 에서 차이가 0 이 아니고 k→0⁻ 로 하강하면 사전 프로그램 급락(Ashenfelter')
    print('dip) — ML 결론(이력 학습가능성 ~1%)과 패널 계량이 같은 선발 구조를 가리킵니다.')
REPORT['117C16 사건연구'] = es_report(C117a)
step('[117C16] 사건연구 r16k')

In [ ]:
%%time
# ═══ [117D16] 선택 전이 (b)단계 — real Qini 를 r16k 에서 (14후보 × 5시드) ═══
D117a, _qc16 = transfer_block(CO16, F16, SEEDS_16K, '117D_후보표_r16k.csv')
T117a = transfer_test(D117a[np.isfinite(D117a[_qc16])], _qc16)
save_csv(T117a, '117D_전이검정_r16k.csv')
print(); print(D117a.to_string(index=False)); print(); print(T117a.to_string(index=False))
print()
print('읽는 법: √PEHE 의 Spearman 이 음수 + 순열 p 유의 + LOO 범위가 0 을 안 넘으면,')
print('「지표 역전」은 후보 몇 개의 우연이 아니라 풀 전체의 성질 — 저널급 증거입니다.')
print('주의: √PEHE 는 낮을수록 좋음 — 음의 Spearman = 지표 양호할수록 실데이터 가치 열위.')
REPORT['117D16 전이'] = '√PEHE %.2f(p=%.3f) · ρ %.2f(p=%.3f) · 후보 %d' % (
    T117a.loc[0, 'Spearman'], T117a.loc[0, '순열p'],
    T117a.loc[1, 'Spearman'], T117a.loc[1, '순열p'], len(D117a))
step('[117D16] 선택 전이 r16k')

In [ ]:
%%time
# ═══ [117E16] 지평 전개 — r16k 실코호트 행 (HGB·RF·Ridge × 3시드) ═══
E117a = have('117E_지평_r16k.csv', '117E16')
if E117a is None:
    _reps16 = [('HGB-T d6', lambda s=0: _HGB(max_depth=6, max_iter=300, random_state=s)),
               ('RF-T 400', lambda s=0: _RF(n_estimators=400, min_samples_leaf=5,
                                            random_state=s, n_jobs=-1)),
               ('Ridge-T',  lambda s=0: _MP(_SS(), _RD(alpha=1.0)))]
    E117a = horizons_block(CO16, F16, _reps16, (0, 1, 2))
    save_csv(E117a, '117E_지평_r16k.csv')
print(E117a.to_string(index=False))
REPORT['117E16 지평'] = ' · '.join('h=%d %s %.0f' % (r['지평'], str(r['후보'])[:3], r['Qini_r16k'])
                                   for _, r in E117a.iterrows())
step('[117E16] 지평 r16k')

In [ ]:
%%time
# ═══ [117SW] 실코호트 2부: r16k 해제 → r240k 로드 (국가규모 승격) — 로드 전용 ═══
_why240 = None
if not RUN_240K:
    _why240 = 'RUN_240K=False'
elif not os.path.exists(_cfg.RAW.get('empservicereal_r240k', '')):
    _why240 = 'datareal/empservicereal_r240k.parquet 없음'
elif not std_available('r240k'):
    _why240 = ('표준화본(npz) 없음 — [117MAKE] 셀이 안 돌았거나 실패했습니다. '
               '[117MAKE] 를 실행해 만든 뒤 여기서부터 이어 실행하십시오')

_HAS240 = _why240 is None
if not _HAS240:
    print('[117SW] r240k 를 건너뜁니다 —', _why240)
    REPORT['117SW'] = 'r240k 건너뜀 (%s)' % _why240.split(' — ')[0]
else:
    for _v in ('CO16', 'F16'):                     # 상태배열 해제 후 240k 로드 (메모리)
        globals().pop(_v, None)
    gc.collect()
    CO24 = load_cohort('r240k')
    REPORT['117SW'] = 'r240k 로드 (N=%s)' % format(CO24['N'], ',')
step('[117SW] 코호트 전환')

In [ ]:
%%time
# ═══ [117C24] 사건연구 — r240k (설계 동일 · CI 는 국가규모 정밀도) ═══
if not _HAS240:
    print('[117C24] 건너뜀 (r240k 없음)')
else:
    C117b = have('117C_사건연구_r240k.csv', '117C24')
    if C117b is None:
        C117b = event_study(CO24)
        save_csv(C117b, '117C_사건연구_r240k.csv')
        print(C117b[C117b['판'] == '사전평균제거'].round(2).to_string(index=False))
    REPORT['117C24 사건연구'] = es_report(C117b)
    step('[117C24] 사건연구 r240k')

In [ ]:
%%time
# ═══ [117E24] 지평 전개 — r240k (HGB·Ridge × 3시드 — RF 400 은 240k 비용상 제외) ═══
if not _HAS240:
    print('[117E24] 건너뜀 (r240k 없음)')
else:
    if CO24.get('s') is not None:                  # (재실행 안전: 이미 해제했으면 F24 재사용)
        F24 = mdl.flatten_features(CO24['s'], CO24['o'])
        CO24['s'] = None; gc.collect()             # 상태배열(~8GB) 해제 — 이후는 F24 만 사용
    E117b = have('117E_지평_r240k.csv', '117E24')
    if E117b is None:
        _reps24 = [('HGB-T d6', lambda s=0: _HGB(max_depth=6, max_iter=300, random_state=s)),
                   ('Ridge-T',  lambda s=0: _MP(_SS(), _RD(alpha=1.0)))]
        E117b = horizons_block(CO24, F24, _reps24, SEEDS_240K)
        save_csv(E117b, '117E_지평_r240k.csv')
    print(E117b.to_string(index=False))
    REPORT['117E24 지평'] = ' · '.join('h=%d %s %.0f' % (r['지평'], str(r['후보'])[:3], r['Qini_r240k'])
                                       for _, r in E117b.iterrows())
    step('[117E24] 지평 r240k')

In [ ]:
%%time
# ═══ [117F24] 선택 전이 — r240k (14후보 × 3시드 · 후보마다 저장 · 이어돌기) ═══
# 가장 긴 셀입니다 (3–7시간). 끊겨도 117F_후보표_r240k.csv 에 끝난 후보가 남고,
# 재실행하면 이어서 돕니다. 시간이 없으면 SKIP_240K_CANDS 에 kNN 둘을 넣으십시오.
if not (_HAS240 and RUN_240K_TRANSFER):
    print('[117F24] 건너뜀 (%s)' % ('r240k 없음' if not _HAS240 else 'RUN_240K_TRANSFER=False'))
else:
    D117b, _qc24 = transfer_block(CO24, F24, SEEDS_240K, '117F_후보표_r240k.csv',
                                  skip=SKIP_240K_CANDS)
    _D24v = D117b[np.isfinite(D117b[_qc24])]
    T117b = transfer_test(_D24v, _qc24); T117b.insert(0, '코호트', 'r240k')
    # 교차규모 비교 — 같은 후보 집합으로 r16k 쪽도 다시 계산해 나란히 둡니다
    _D16x = pd.read_csv(os.path.join(DIR_OUT, '117D_후보표_r16k.csv'), encoding='utf-8-sig')
    _D16v = _D16x[_D16x['후보'].isin(_D24v['후보'])]
    T16c = transfer_test(_D16v, 'real_Qini_r16k'); T16c.insert(0, '코호트', 'r16k(동일후보)')
    TX = pd.concat([T16c, T117b], ignore_index=True)
    save_csv(TX, '117F_전이검정_교차규모.csv')
    print(); print(D117b.to_string(index=False)); print(); print(TX.to_string(index=False))
    REPORT['117F24 전이'] = 'r240k: √PEHE %.2f(p=%.3f) · ρ %.2f(p=%.3f) · 후보 %d/%d' % (
        T117b.loc[0, 'Spearman'], T117b.loc[0, '순열p'],
        T117b.loc[1, 'Spearman'], T117b.loc[1, '순열p'], len(_D24v), len(D117b))
    step('[117F24] 선택 전이 r240k')

In [ ]:
# ═══ [117T] 마무리 ═══
_outs = sorted(f for f in os.listdir(DIR_OUT) if f.startswith('117'))
print('산출물 (out/):'); [print('  ', f) for f in _outs]
print()
for _k in sorted(REPORT):
    print('%-16s %s' % (_k, REPORT[_k]))
print()
print('→ out/117* 를 보내 주시면 전체(시뮬레이터 + 실코호트 r16k·r240k) 검증을 마치고')
print('   저널 원고 집필에 들어갑니다. 이 노트북 하나가 run111~116 을 전부 대체하므로')
print('   구 노트북들과 out/111*·113*·114*·115*·116* 파일은 삭제해도 됩니다.')
step('[117T] run117 완료')

## End